In [34]:
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
import os
google_api_key = os.getenv("GOOGLE_API_KEY")

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

In [37]:
loader =PyPDFLoader("../data/dermAI.pdf")
docs = loader.load()
len(docs)

32

In [38]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

34

In [39]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [40]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [41]:
query= "machine learning, deep learning and langchain and AI Agents"
data = vector_store.similarity_search(query=query)

In [42]:
len(data)

4

In [43]:
data[0]

Document(id='69c0a71f-d0f2-4ece-811f-5c416bb4df1d', metadata={'total_pages': 32, 'source': '../data/dermAI.pdf', 'page_label': '1', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'moddate': '2026-10-07T11:16:46+00:00', 'title': 'DermAI: Skin Disease Detector from Mobile Images Using DL', 'creationdate': '2026-10-07T11:16:46+00:00', 'producer': 'Skia/PDF m128', 'page': 0}, page_content='DermAI: Skin Disease Detector\nfrom Mobile Images Using DL\n1. Title\nDermAI\x00 Skin Disease Detector from Mobile Images Using Deep Learning\n2. Objective\nBuild an AI-powered dermatological image analysis system that uses mobile\nphone images of skin lesions and skin conditions to automatically identify,\nclassify, and analyze potential skin diseases using deep learning techniques.\nThe platform will allow users to capture or upload an image of a skin lesion\nthrough a mobile or web application. The system will preprocess the 

In [44]:
context = ""
for doc in data:
    context += doc.page_content+ "\n"

In [45]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [46]:
res =llm.invoke(f"Can You Provide me the answer based on the context for my question, context : {context} and question : {query}")

In [47]:
print(res.content)

Below is a concise, **context‑driven** answer that ties together the three topics you mentioned – **Machine Learning (ML), Deep Learning (DL) and LangChain / AI Agents** – with the **DermAI: Skin Disease Detector** project you described.

---

## 1. Where ML & DL already live in DermAI  

| Pipeline Stage | Typical ML/DL Technique | Why it’s used | Example model from your list |
|----------------|------------------------|---------------|------------------------------|
| **Image acquisition & quality check** | Classical ML (e.g., SVM / Random Forest) on handcrafted quality metrics (blur, illumination, color histogram) | Quickly discard unusable pictures before heavy DL inference | None required – can be a lightweight scikit‑learn classifier |
| **Lesion / region detection** | **Object detection** (Mask R‑CNN) or **semantic segmentation** (U‑Net, DeepLabV3) | Localise the skin patch that actually contains the lesion | Mask R‑CNN, U‑Net++, Attention U‑Net |
| **Feature extraction & classi

### CHAIN - CONTEXT GENERATE | PROMPT|LLM | STRPARSER

In [68]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content+ "\n"
    return{
        "context" :context,
        "question": query
    
    }

In [69]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the user's question based only on the provided context.

If you don't know the answer from the context, simply say:
"I Dont Know The Answer Of This Now!!!! Sorryyyy!!!!!!!"

Context:
{context}

Question:
{question}
""")

In [70]:
rag_chain = get_context | prompt | llm

In [73]:
res = rag_chain.invoke("what is OS")

In [74]:
print(res.content)

I Dont Know The Answer Of This Now!!!! Sorryyyy!!!!!!!
